## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: MANTENIMIENTO Y CARACTERÍSTICAS AVANZADAS DELTA LAKE
**OBJETIVO:** Demostrar el uso de Time Travel, OPTIMIZE y VACUUM en la tabla Silver

In [0]:
from pyspark.sql.functions import col

dbutils.widgets.removeAll()
dbutils.widgets.text("catalogo", "catalogo_pqrs")
dbutils.widgets.text("esquema", "silver")
dbutils.widgets.text("tabla", "tickets_enriquecidos")

catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
tabla = dbutils.widgets.get("tabla")

ruta_tabla = f"{catalogo}.{esquema}.{tabla}"
print(f"✅ Apuntando a la tabla: {ruta_tabla}")

### 1. TIME TRAVEL E HISTORIAL
Vamos a ver el registro de auditoría de todo lo que le ha pasado a esta tabla.

In [0]:
# Ver el historial de versiones de la tabla

display(spark.sql(f"DESCRIBE HISTORY {ruta_tabla}"))

In [0]:
# Simulamos un accidente: Borrar todos los tickets de un agente específico

spark.sql(f"DELETE FROM {ruta_tabla} WHERE agente_asignado = 'Carlos Ramirez'")
print("⚠️ Tickets de Carlos Ramirez eliminados accidentalmente.")

In [0]:
# ¡TIME TRAVEL AL RESCATE!

# Restauramos la tabla a su versión 0 (antes de que borraramos los datos)
spark.sql(f"RESTORE TABLE {ruta_tabla} TO VERSION AS OF 0")
print("✅ Tabla restaurada exitosamente a la Versión 0.")

### 2. OPTIMIZACIÓN (Z-ORDER)
Reorganizamos los archivos físicos para acelerar las consultas filtradas por región.

In [0]:
print("Optimizando tabla...")
spark.sql(f"OPTIMIZE {ruta_tabla} ZORDER BY (region_agente)")
print("✅ Optimización completada.")

### 3. VACUUM (ASPIRADORA)
Eliminamos el historial de archivos basura. Por seguridad, Databricks no deja borrar historial menor a 7 días, pero lo forzaremos a 0 horas solo para la demostración.

In [0]:
# Pasamos la aspiradora para limpiar archivos obsoletos con la retención mínima permitida en serverless
spark.sql(f"VACUUM {ruta_tabla} RETAIN 168 HOURS")
print("✅ Limpieza Vacuum ejecutada.")